# 서울 사고 데이터 전처리

2023~2025년 서울 사고누적지점의 사망·중상사고를 분석한다. 위에서 아래로 실행하면서 원본의 첫 5행, 지점별 검증 결과, 열을 선택한 결과, 저장된 CSV를 차례로 확인한다.

## 1. 원본 읽기
TMACS 2025년 기준 사고상세 CSV를 문자열로 읽어 사고번호의 자릿수와 빈칸을 보존한다. 발생연도별 건수와 첫 5행에서 사고번호, 지점명, 발생 시각을 확인한다.

In [1]:
import csv
from collections import Counter
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/tmacs/2025/사고상세.csv").is_file())
raw = root / "data/raw/tmacs/2025"
output = root / "data/processed/사고_분석용.csv"

def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

rows = read_csv(raw / "사고상세.csv")
years = Counter(row["연도"] for row in rows)
assert set(years) == {"2023", "2024", "2025"}
print("발생연도별 사고 수", dict(sorted(years.items())))

from html import escape
from IPython.display import HTML, display

def show_first_five(records, fields):
    headers = "".join(f"<th>{escape(field)}</th>" for field in fields)
    body = []
    for record in records[:5]:
        cells = "".join(f"<td>{escape(str(record[field]))}</td>" for field in fields)
        body.append(f"<tr>{cells}</tr>")
    display(HTML(f"<table><thead><tr>{headers}</tr></thead><tbody>{''.join(body)}</tbody></table>"))

show_first_five(rows, ["사고번호", "지점ID", "지점명", "연도", "월", "일", "시각"])

발생연도별 사고 수 {'2023': 1539, '2024': 1462, '2025': 1563}


사고번호,지점ID,지점명,연도,월,일,시각
2023011100100197,202500111680_001n,수서역앞교차로,2023,01,11,11
2023021100100329,202500111680_001n,수서역앞교차로,2023,02,11,18
2023061500100097,202500111680_001n,수서역앞교차로,2023,06,15,08
2023060800100215,202500111680_001n,수서역앞교차로,2023,06,08,12
2023073000100220,202500111680_001n,수서역앞교차로,2023,07,30,14


## 2. 지점별 사고 건수와 사고번호 검증
지점마다 목록상 사고 건수가 상세 행 수와 같은지 확인한다. 전체 사고번호의 누락과 중복도 검사한다. 아래 표에서 처음 5개 지점의 두 건수를 나란히 확인할 수 있다.

In [2]:
counts = Counter(row["지점ID"] for row in rows)
sites = read_csv(raw / "지점목록.csv")
assert counts == {s["지점ID"]: int(s["지점사고건수"]) for s in sites}
assert all(counts[r["지점ID"]] == int(r["지점사고건수"]) for r in rows)
ids = [r["사고번호"] for r in rows]
assert all(len(i) == 16 and i.isdigit() for i in ids)
assert len(set(ids)) == len(rows)
print(f"지점 {len(counts)}개, 사고 {len(ids):,}건, 사고번호 누락·중복 0건")

site_checks = [
    {"지점명": site["지점명"], "목록상 건수": site["지점사고건수"], "상세 행 수": counts[site["지점ID"]]}
    for site in sites
]
show_first_five(site_checks, ["지점명", "목록상 건수", "상세 행 수"])

지점 327개, 사고 4,564건, 사고번호 누락·중복 0건


지점명,목록상 건수,상세 행 수
수서역앞교차로,27,27
학여울역,13,13
대치우성아파트사거리,14,14
대치역,10,10
삼성역,15,15


## 3. 분석용 열 선택과 시각 정리
사고 한 건당 한 행을 유지하면서 지점 중심 좌표와 사고 특성 등 26개 열을 선택한다. 시각은 두 자리로 통일하고 나머지 값과 빈칸은 원본대로 둔다. 첫 5행에서 원본 시각과 변환한 시각을 비교한다.

In [3]:
columns = [
    "자치구", "지점명", "지점위도", "지점경도", "연도", "월", "일", "시각",
    "사고등급", "사망자수", "중상자수", "경상자수", "부상자수",
    "제1당사자_연령", "제1당사자_성별", "제1당사자_운전경력",
    "제1당사자_차종", "제1당사자_차량용도", "관련자2", "사고형태", "기상상태",
    "일광상태", "사고유형_대분류", "사고유형_세부", "법규위반", "도로종류",
]

analysis = []
for row in rows:
    selected = {column: row[column] for column in columns}
    if selected["시각"]:
        hour = int(selected["시각"])
        assert 0 <= hour <= 23
        selected["시각"] = f"{hour:02d}"
    analysis.append(selected)

print(f"분석용 자료: {len(analysis):,}행 × {len(columns)}열")

time_checks = [
    {"지점명": before["지점명"], "원본 시각": before["시각"], "분석용 시각": after["시각"]}
    for before, after in zip(rows[:5], analysis[:5])
]
show_first_five(time_checks, ["지점명", "원본 시각", "분석용 시각"])

분석용 자료: 4,564행 × 26열


지점명,원본 시각,분석용 시각
수서역앞교차로,11,11
수서역앞교차로,18,18
수서역앞교차로,08,08
수서역앞교차로,12,12
수서역앞교차로,14,14


## 4. 분석용 CSV 저장
결과를 `data/processed/사고_분석용.csv`에 저장한다. 기존 파일의 열 순서와 값이 같으면 그대로 사용한다. 저장 경로에서 다시 읽은 값이 메모리의 전처리 결과와 일치하는지 확인한다.

In [4]:
output = root / "data/processed/사고_분석용.csv"
output.parent.mkdir(parents=True, exist_ok=True)
if not output.exists() or read_csv(output) != analysis or list(read_csv(output)[0]) != columns:
    temporary = output.with_suffix(".csv.tmp")
    try:
        with temporary.open("w", encoding="utf-8-sig", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=columns)
            writer.writeheader()
            writer.writerows(analysis)
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)
assert read_csv(output) == analysis
print(f"수정본 확인: {output.name} · {len(analysis):,}행")

수정본 확인: 사고_분석용.csv · 4,564행

## 5. 저장된 결과 확인
분석용 CSV에서 첫 5행을 다시 읽어 26개 열의 순서와 값을 확인한다.

In [5]:
show_first_five(read_csv(output), columns)

자치구,지점명,지점위도,지점경도,연도,월,일,시각,사고등급,사망자수,중상자수,경상자수,부상자수,제1당사자_연령,제1당사자_성별,제1당사자_운전경력,제1당사자_차종,제1당사자_차량용도,관련자2,사고형태,기상상태,일광상태,사고유형_대분류,사고유형_세부,법규위반,도로종류
강남구,수서역앞교차로,37.487374913895344,127.10170388356964,2023,01,11,11,중상사고,0,1,0,0,47,여,15년이상,승용,비사업용,이륜,일직선사고,맑음,주간,차대차,차대차-기타,중앙선침범,특별광역시도
강남구,수서역앞교차로,37.487374913895344,127.10170388356964,2023,02,11,18,중상사고,0,1,0,0,24,남,10년미만,승용,비사업용,승용,진입회전사고,맑음,출퇴근,차대차,측면직각충돌,안전거리미확보,특별광역시도
강남구,수서역앞교차로,37.487374913895344,127.10170388356964,2023,06,15,08,중상사고,0,1,1,0,65,남,15년이상,승용,사업용,승용,일직선사고,맑음,출퇴근,차대차,추돌(진행중),안전거리미확보,특별광역시도
강남구,수서역앞교차로,37.487374913895344,127.10170388356964,2023,06,08,12,중상사고,0,1,0,0,74,여,15년이상,승용,비사업용,승용,일직선사고,흐림,주간,차대차,추돌(진행중),안전운전불이행,특별광역시도
강남구,수서역앞교차로,37.487374913895344,127.10170388356964,2023,07,30,14,중상사고,0,1,0,0,48,남,15년이상,화물,비사업용,화물,기타사고,맑음,주간,차대차,차대차-기타,안전거리미확보,특별광역시도


## 6. 빈칸과 미상 표기 점검
26개 분석 열에서 빈칸, 데이터없음, 불명을 구분해 센다. 한 사고에서 여러 열이 비어 있을 수 있으므로 열별 건수를 더해 사고 수로 해석하지 않는다. 성별·연령 미상 사고는 그대로 보존하고, 성별이나 연령별 비중을 낼 때 미상 건수를 함께 제시한다.

In [6]:
def missing_label(value):
    label = "".join(value.split())
    return {"": "빈칸", "데이터없음": "데이터없음", "불명": "불명"}.get(label)

age = "제1당사자_연령"
sex = "제1당사자_성별"
for column in columns:
    counts_by_label = Counter(missing_label(row[column]) for row in analysis)
    blank, absent, unknown = (counts_by_label[label] for label in ("빈칸", "데이터없음", "불명"))
    if blank + absent + unknown or column in (age, sex):
        print(f"{column}: 빈칸 {blank:,}, 데이터없음 {absent:,}, 불명 {unknown:,}")

age_missing = {i for i, row in enumerate(analysis) if missing_label(row[age])}
sex_missing = {i for i, row in enumerate(analysis) if missing_label(row[sex])}
print(f"전체 {len(analysis):,}건 중 연령 미상 {len(age_missing):,}건, 성별 미상 {len(sex_missing):,}건")
print(f"둘 다 미상 {len(age_missing & sex_missing):,}건, 하나 이상 미상 {len(age_missing | sex_missing):,}건")
examples = [analysis[i] for i in sorted(age_missing | sex_missing)]
show_first_five(examples, ["지점명", "연도", age, sex])

제1당사자_연령: 빈칸 0, 데이터없음 0, 불명 49


제1당사자_성별: 빈칸 0, 데이터없음 0, 불명 49
제1당사자_차량용도: 빈칸 0, 데이터없음 779, 불명 0
전체 4,564건 중 연령 미상 49건, 성별 미상 49건
둘 다 미상 49건, 하나 이상 미상 49건


지점명,연도,제1당사자_연령,제1당사자_성별
학여울역,2023,불명,불명
강남세브란스사거리,2024,불명,불명
한양아파트앞사거리,2024,불명,불명
강남소방서영동119안전센터 북동쪽 교차로,2024,불명,불명
역삼역사거리,2024,불명,불명


## 음주 열 제외와 미상 값 보존 확인
원본에서 제1당사자 음주 값이 모두 빈칸인지 확인한다. 분석 자료에는 이 열만 추가로 제외하고, 성별·연령과 차량용도의 미상 표기를 행별로 보존한다.

In [7]:
assert all(not row["제1당사자_음주"].strip() for row in rows)
assert "제1당사자_음주" not in columns
assert len(rows) == len(analysis)
for original, processed in zip(rows, analysis):
    for field in ("제1당사자_성별", "제1당사자_연령", "제1당사자_차량용도"):
        assert original[field] == processed[field]
both_unknown = sum(r["제1당사자_성별"] == r["제1당사자_연령"] == "불명" for r in analysis)
vehicle_unknown = sum(r["제1당사자_차량용도"] == "데이터없음" for r in analysis)
print(f"음주 열 제외 · 성별·연령 모두 불명 {both_unknown}건 · 차량용도 데이터없음 {vehicle_unknown}건")

음주 열 제외 · 성별·연령 모두 불명 49건 · 차량용도 데이터없음 779건
